# Train the notes model (QLoRA)

Fine-tunes Llama 3.2 3B Instruct on the 150 dental notes with QLoRA (Hugging Face PEFT + Transformers).

Before running:
1. Runtime > Change runtime type > T4 GPU
2. Upload `notes_dataset.jsonl` when the upload cell runs
3. Llama 3.2 is gated: accept the license at huggingface.co/meta-llama/Llama-3.2-3B-Instruct and paste your HF token when asked

If a pinned version conflicts with Colab, bump it and re-run the install cell.

In [ ]:
!nvidia-smi
import torch
print("cuda available:", torch.cuda.is_available())

In [ ]:
!pip install -q transformers==4.46.2 peft==0.13.2 trl==0.12.1 bitsandbytes==0.45.5 accelerate==1.1.1 datasets==3.1.0

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick notes_dataset.jsonl

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

rows = []
with open("notes_dataset.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print("loaded", len(rows), "examples")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_text(row):
    messages = [
        {"role": "system", "content": row["instruction"]},
        {"role": "user", "content": row["input"]},
        {"role": "assistant", "content": json.dumps(row["output"])},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

dataset = Dataset.from_list([{"text": to_text(r)} for r in rows])
print(dataset[0]["text"][:300])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="notes_lora_out",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    max_seq_length=1024,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    peft_config=lora_config,
)
trainer.train()

In [ ]:
trainer.model.save_pretrained("notes_lora_adapter")
tokenizer.save_pretrained("notes_lora_adapter")

!zip -r notes_lora_adapter.zip notes_lora_adapter
from google.colab import files
files.download("notes_lora_adapter.zip")